# Claculated Values
- Here are all the Caclulated values needed for Power BI
- All sites will have the same Excel Format


## Checkin 

* ZzAge: Continous variable for number of business days since submission.
* ZzCheckAge: Continous Variable for dates from Visit to when checked in.
* ZzCriticalCount: Continous variable, counts of number of critcal findings.
* ZzFindingsCount: Continous variable, counts of number of findings for a chart.
* ZzMajorCount: Continous variable, counts the number of majot findings.
* ZzNewest: Binary variable for latest submission chart checkin submission for a Subject/Visit.
* ZzPriority: **Measure** Categorical variable, Yellow when ZzAge eq 2 and Red when gt 2.
* ZzQCAge: Continous Variable for dates from Check In to when QC'd.
* ZzSignifigance: **Measure** Continous varaible, counts types of findings per visit.
* ZzStatus: Binary variable for if chart is checked in or checked out.
* ZzTimeStamp: Date variable to combine date and time. 

* ZzTurnAround: Continous variable, counts the number fo days to complete a visit.
* ZzTurnAroundCount: **Measure** Continous variable, counts the number of charts that take less than 3 days to complete.
* ZzTurnAroundPCT: **Measure** Continous, percent of charts done under 2days minus the ones checked out by clinical stafff.
* ZzWeekofCheckIn: Categorical variable, group of days by work week.


```dax

ZzAge = 
    COUNTROWS(
        FILTER(
            'CalendarTable',
            'CalendarTable'[Date] >= [Check-In Date] &&
            'CalendarTable'[Date] <= TODAY() &&
            'CalendarTable'[IsBusinessDay] = TRUE
        )
    ) - 1


ZzCheckAge = 
IF(
    ISBLANK(RELATED(CTMS[Visit By:Date])),
    DATEDIFF(TODAY(), RELATED(CTMS[Visit By:Date]), DAY),
    DATEDIFF(CheckIn[Check-In Date], RELATED(CTMS[Visit By:Date]), DAY)
)

ZZCriticalCount = 
IF(
    ISBLANK(
        CALCULATE(
            COUNTROWS(Findings1),
            FILTER(
                Findings1, 
                Findings1[Visit Key] = EARLIER(CheckIn[Visit Key]) &&
                Findings1[Follow Up] = "Critical - Follow Up"
            )
        )
    ),
    0,
    CALCULATE(
        COUNTROWS(Findings1),
        FILTER(
            Findings1, 
            Findings1[Visit Key] = EARLIER(CheckIn[Visit Key]) &&
            Findings1[Follow Up] = "Critical - Follow Up"
        )
    )
)



ZzFindingsCount = 
IF(
    ISBLANK(
        CALCULATE(
            COUNTROWS(Findings1),
            FILTER(
                Findings1, 
                Findings1[Visit Key] = EARLIER(CheckIn[Visit Key]) &&
                NOT (Findings1[Category] = "None" || Findings1[Category] = "All Good")
            )
        )
    ),
    0,
    CALCULATE(
        COUNTROWS(Findings1),
        FILTER(
            Findings1, 
            Findings1[Visit Key] = EARLIER(CheckIn[Visit Key]) &&
            NOT (Findings1[Category] = "None" || Findings1[Category] = "All Good")
        )
    )
)




ZZMajorCount = 
IF(
    ISBLANK(
        CALCULATE(
            COUNTROWS(Findings1),
            FILTER(
                Findings1, 
                Findings1[Visit Key] = EARLIER(CheckIn[Visit Key]) &&
                Findings1[Follow Up] = "Major - Follow Up"
            )
        )
    ),
    0,
    CALCULATE(
        COUNTROWS(Findings1),
        FILTER(
            Findings1, 
            Findings1[Visit Key] = EARLIER(CheckIn[Visit Key]) &&
            Findings1[Follow Up] = "Major - Follow Up"
        )
    )
)


ZzNewest = 
    VAR MaxTimeSubmitted =
        CALCULATE(
            MAX(CheckIn[ZzTimeStamp]),
            ALLEXCEPT(CheckIn, 'CheckIn'[Study], 'CheckIn'[Subject Number], 'CheckIn'[Visit])
        )
    RETURN IF('CheckIn'[ZzTimeStamp] = MaxTimeSubmitted, 1, 0)



ZzPriority = 
VAR AgeValue = SELECTEDVALUE('CheckIn'[ZzAge])
RETURN 
    IF(AgeValue = 2, "#FFFF00", IF(AgeValue > 2, "#FF0000", BLANK()))


ZzQCAge = 
IF(
    ISBLANK(Related(DEOut[QC Date])),
    DATEDIFF(TODAY(), Related(DEOut[QC Date]), DAY),
    DATEDIFF(CheckIn[Check-In Date], RELATED(DEOut[QC Date]), DAY)
)



ZzSignificance = 
VAR VisitKey = SELECTEDVALUE(CheckIn[Visit Key])
VAR FollowUpValues = 
    CALCULATETABLE(
        VALUES('Findings1'[Follow Up]),
        'Findings1'[Visit Key] = VisitKey
    )
VAR HasCritical = CONTAINSSTRING(CONCATENATEX(FollowUpValues, [Follow Up], ","), "Critical")
VAR HasMajor = CONTAINSSTRING(CONCATENATEX(FollowUpValues, [Follow Up], ","), "Major")
VAR HasNormal = CONTAINSSTRING(CONCATENATEX(FollowUpValues, [Follow Up], ","), "Normal")

RETURN
IF(
    COUNTROWS(FollowUpValues) = 0,
    "Pending",
    IF(
        HasCritical && HasMajor,
        "Critical and Major",
        IF(
            HasCritical,
            "Critical",
            IF(
                HasMajor,
                "Major",
                IF(
                    HasNormal,
                    "Normal",
                    "Pending"
                )
            )
        )
    )
)

ZzStatus = 
IF (
    NOT ( ISBLANK ( RELATED ( 'DEOut'[Visit Key]) ) )
        || NOT ( ISBLANK ( RELATED ('CheckOut'[Visit Key]) ) ),
    "Out",
    "In"
)   



ZzTimeStamp = 
    VAR TimeStampString = CheckIn[Check-In Date] & " " & CheckIn[Check In Time]
    RETURN DATEVALUE(TimeStampString) + TIMEVALUE(TimeStampString)


ZzTurnAround = 
VAR CheckInDate = 'CheckIn'[Check-In Date]
VAR QCDate = RELATED('DEOut'[QC Date])
VAR EndDate = IF(ISBLANK(QCDate), TODAY(), QCDate)
RETURN
    COUNTROWS(
        FILTER(
            'CalendarTable',
            'CalendarTable'[Date] >= CheckInDate &&
            'CalendarTable'[Date] <= EndDate &&
            'CalendarTable'[IsBusinessDay] = TRUE()
        )
    ) - 1



ZzTurnAroundCount = 
COUNTROWS(
    FILTER(
        CheckIn,
        CheckIn[ZzTurnAround] < 3
    )
)



ZzTurnAroundPCT = 
DIVIDE(
    [ZzTurnAroundCount] - COUNT('CheckOut'[Visit Key]), 
    COUNTA('CheckIn'[ZzTurnAround]) - COUNT('CheckOut'[Visit Key])
)


ZzWeekofCheckIn = 
VAR StartOfWeek = 'Checkin'[Check-in Date] - WEEKDAY('Checkin'[Check-in Date], 2) + 1  -- Monday
VAR EndOfWeek = StartOfWeek + 4  -- Friday (Monday + 4 days)
RETURN FORMAT(StartOfWeek, "dd-mmm-yyyy") & " - " & FORMAT(EndOfWeek, "dd-mmm-yyyy")











```

## DEOut

* ZzChartTime: Continous Variable, time diffrence between the Start of a chart and DeOut 
* ZzPendingCheckIn: Continous Varaible, count of charts pending QC
* ZzPlogMetric: Continous Variable, Count of Findings per Visit
* ZzWeekOfQC:Categorical Variable, group of days for a given week.

```dax

ZzCharttime = 
IF(
    ISBLANK(RELATED(ChartStart[Visit Key])) || ISBLANK(DEOut[QC Time]) || ISBLANK(RELATED(ChartStart[Start By:Time])),
    0,
    ABS(
        (HOUR(RELATED(ChartStart[Start By:Time])) * 60 + MINUTE(RELATED(ChartStart[Start By:Time]))) - 
        (HOUR(DEOut[QC Time]) * 60 + MINUTE(DEOut[QC Time]))
    )
)

ZzPendingCheckIn = COUNTA('CheckIn'[Visit Key]) - COUNTA('DEOut'[Visit Key])

ZzPlogMetric = 
VAR VisitKey = DEOut[Visit Key]
VAR VisitCount = 
    CALCULATE(
        COUNTROWS(Findings1),
        Findings1[Visit Key] = VisitKey,
        Findings1[Category] <> "None"  -- Filter out rows where Category is "None"
    )
RETURN
    IF(ISBLANK(VisitCount), 0, VisitCount)


ZzWeekOfQC = 
VAR StartOfWeek = 'DEOut'[QC Date] - WEEKDAY('DEOut'[QC Date], 2) + 1  -- Monday
VAR EndOfWeek = StartOfWeek + 4  -- Friday (Monday + 4 days)
RETURN FORMAT(StartOfWeek, "dd-mmm-yyyy") & " - " & FORMAT(EndOfWeek, "dd-mmm-yyyy")


```


## Findings

* ZzCompletion Time: Continous variable for time between Findings issued and findings verified.

* ZzResponse Time: Continous variable for time between Findings issued and finding Responded to by Clinical.

* ZzStatus: Categorical variable, if findings in pending Response or has been responded to. 

* ZzUrgent: **Measure** Categorical variable, Yellow/Red.

* ZzWeekofFindings: Categorical Varaible, group of dates by week.

* ZzXcount: Continous variable for number of yes values in the `Findings[Follow up]` column. 







```dax
ZzCompletion Time = 
IF(
    ISBLANK(RELATED('Response'[Completed By:Date])),
    COUNTROWS(
        FILTER(
            'CalendarTable',
            'CalendarTable'[Date] >= 'Findings1'[Review Date] &&
            'CalendarTable'[Date] <= TODAY() &&
            'CalendarTable'[IsBusinessDay] = TRUE
        )
    ) - 1,
    IF(
        ISBLANK(RELATED('Verify'[Verified By:Date])),
        COUNTROWS(
            FILTER(
                'CalendarTable',
                'CalendarTable'[Date] >= 'Findings1'[Review Date] &&
                'CalendarTable'[Date] <= TODAY() &&
                'CalendarTable'[IsBusinessDay] = TRUE
            )
        ) - 1,
        COUNTROWS(
            FILTER(
                'CalendarTable',
                'CalendarTable'[Date] >= 'Findings1'[Review Date] &&
                'CalendarTable'[Date] <= RELATED('Verify'[Verified By:Date]) &&
                'CalendarTable'[IsBusinessDay] = TRUE
            )
        )
    )
)



ZzResponse Time = 
IF(
    ISBLANK(RELATED('Response'[Completed By:Date])),
    COUNTROWS(
        FILTER(
            'CalendarTable',
            'CalendarTable'[Date] >= 'Findings1'[Review Date] &&
            'CalendarTable'[Date] <= TODAY() &&
            'CalendarTable'[IsBusinessDay] = TRUE
        )
    ) - 1,
    COUNTROWS(
        FILTER(
            'CalendarTable',
            'CalendarTable'[Date] >= 'Findings1'[Review Date] &&
            'CalendarTable'[Date] <= RELATED('Response'[Completed By:Date]) &&
            'CalendarTable'[IsBusinessDay] = TRUE
        )
    ) - 1
)


ZzStatus = 
    IF(
        ISBLANK(RELATED('Verify'[Verified By:Date])),
        IF(
            ISBLANK(RELATED('Response'[Completed By:Date])),
            "Pending",
            "Responded"
        ),
        "Completed"
    )

ZzUrgent = 
SWITCH(
    TRUE(),
    SELECTEDVALUE('Findings1'[Follow Up]) = "Normal", BLANK(),
    SELECTEDVALUE('Findings1'[Follow Up]) = "Major", "Yellow",
    SELECTEDVALUE('Findings1'[Follow Up]) = "Critical", "Red",
    BLANK()  
)

ZzWeekofFindings = 
VAR StartOfWeek = 'Findings1'[Review Date] - WEEKDAY('Findings1'[Review Date], 2) + 1  -- Monday
VAR EndOfWeek = StartOfWeek + 4  -- Friday (Monday + 4 days)
RETURN FORMAT(StartOfWeek, "dd-mmm-yyyy") & " - " & FORMAT(EndOfWeek, "dd-mmm-yyyy")


ZzXcount = 
CALCULATE(
    COUNTA('Findings1'[Visit Completed by]),
    'Findings1'[Follow Up] = "Major" || 'Findings1'[Follow Up] = "Critical"
)


```

## Calculated Tables

* Table made for wide date options for all data. 

```dax
CalendarTable = 
ADDCOLUMNS (
    CALENDAR (DATE(2020, 1, 1), DATE(YEAR(TODAY()) + 1, MONTH(TODAY()), DAY(TODAY()))),
    "Year", YEAR([Date]),
    "Month", MONTH([Date]),
    "Day", DAY([Date]),
    "DayOfWeek", WEEKDAY([Date]),
    "DayName", FORMAT([Date], "dddd"),
    "IsWeekday", IF(WEEKDAY([Date], 2) < 6, TRUE, FALSE),
    "IsWeekend", IF(WEEKDAY([Date], 2) >= 6, TRUE, FALSE),
    "IsHoliday", 
        IF (
            [Date] = DATE(YEAR([Date]), 1, 1) ||  // New Year's Day
            [Date] = DATE(YEAR([Date]), 1, 20) ||  // MLK Day 
            [Date] = DATE(YEAR([Date]), 7, 4) ||  // Independence Day
            [Date] = DATE(YEAR([Date]), 12, 24) ||  // Christmas Eve
            [Date] = DATE(YEAR([Date]), 12, 25) ||  // Christmas Day
            [Date] = DATE(YEAR([Date]), 12, 31) ||  // New Year's Eve
            [Date] = DATE(YEAR([Date]), 11, 25) ||  // Thanksgiving 
            [Date] = DATE(YEAR([Date]), 11, 26),  // Day after Thanksgiving 
            TRUE, FALSE
        ),
    "IsBusinessDay", 
        IF (
            IF(WEEKDAY([Date], 2) < 6, TRUE, FALSE) = TRUE() && 
            IF (
                [Date] = DATE(YEAR([Date]), 1, 1) ||  // New Year's Day
                [Date] = DATE(YEAR([Date]), 1, 20) ||  // MLK Day 
                [Date] = DATE(YEAR([Date]), 7, 4) ||  // Independence Day
                [Date] = DATE(YEAR([Date]), 12, 24) ||  // Christmas Eve
                [Date] = DATE(YEAR([Date]), 12, 25) ||  // Christmas Day
                [Date] = DATE(YEAR([Date]), 12, 31) ||  // New Year's Eve
                [Date] = DATE(YEAR([Date]), 11, 25) ||  // Thanksgiving 
                [Date] = DATE(YEAR([Date]), 11, 26),  // Day after Thanksgiving
                TRUE, FALSE
            ) = FALSE(),
            TRUE, 
            FALSE
        )
)


```


## CTMS


* ZzTurnAroundCTMS: Continous variable, counts the number fo days to complete a visit.
* ZzTurnAroundCountCTMS: **Measure** Continous variable, counts the number of charts that take less than 3 days to complete.
* ZzTurnAroundPCT_CTMS: **Measure** Continous, percent of charts done under 2days minus the ones checked out by clinical stafff.

In [ ]:


ZzTurnAroundCountCTMS = 
COUNTROWS(
    FILTER(
        CTMS,
        CTMS[ZzTurnAroundCTMS] < 3
    )
)



ZzTurnAroundCTMS = 
VAR CheckInDate = 'CTMS'[Visit By:Date]
VAR QCDate = RELATED('DEOut'[QC Date])
VAR EndDate = IF(ISBLANK(QCDate), TODAY(), QCDate)
RETURN
    COUNTROWS(
        FILTER(
            'CalendarTable',
            'CalendarTable'[Date] >= CheckInDate &&
            'CalendarTable'[Date] <= EndDate &&
            'CalendarTable'[IsBusinessDay] = TRUE()
        )
    ) - 1


ZzTurnAroundPCT_CTMS = 
DIVIDE(
    [ZzTurnAroundCountCTMS] - COUNT('CheckOut'[Visit Key]), 
    COUNTA('CTMS'[Visit Key]) - COUNT('CheckOut'[Visit Key])
)
